# Мешок слов, TF-IDF и разреженные признаки

Векторизация переводит документ в набор числовых признаков. Здесь исследуются счётчики слов, последовательности токенов, TF-IDF и символьные фрагменты. Словарь и частоты обучаются только по train.

Разреженные матрицы рассматриваются как основной способ хранения. В конце показан ограниченный по памяти вариант представления без обучаемого словаря.

## 1. Словарь и мешок слов

Мешок слов связывает документ с числами появлений выбранных токенов. Порядок слов теряется. Столбец матрицы обозначает один выученный элемент, а не класс темы.

**Данные и общая нормализация**

Восстанавливаются тот же источник и та же необучаемая нормализация. Для подбора словаря используется train после объединения согласованных экспортных повторов. Темы не передаются векторизатору.

In [1]:
# Шаг М3.02.01 — Данные и общая нормализация
from pathlib import Path
from collections import Counter
import re
import html
import unicodedata
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer, HashingVectorizer

def normalize_text(text):
    text=unicodedata.normalize('NFKC',html.unescape(str(text)))
    text=re.sub(r'<[^>]+>',' ',text)
    text=re.sub(r'\d+',' номер ',text)
    return ' '.join(text.casefold().split())

module=Path.cwd()
course_data=next((p/'data' for p in (module,*module.parents) if (p/'data'/'reviews.txt').is_file()),None)
if course_data is None:
    raise FileNotFoundError('Нужен исходный reviews.txt')
messages=pd.read_csv(module/'data'/'support_messages.csv')
assert messages.groupby('case_id')[['text','topic','family_id','split']].nunique().le(1).all().all()
messages=messages.drop_duplicates('case_id').copy()
train=messages.loc[messages['split'].eq('train')].copy()
valid=messages.loc[messages['split'].eq('validation')].copy()
assert set(train['family_id']).isdisjoint(valid['family_id'])
print('Обучающих документов:',len(train),'валидационных:',len(valid))

Обучающих документов: 1452 валидационных: 363


**Матрица счётчиков на четырёх документах**

CountVectorizer совмещает выделение элементов и подсчёт. По умолчанию токен содержит не менее двух буквенно-цифровых символов. Только для такой маленькой матрицы далее показывается плотная таблица.

Документ задаёт строку матрицы, выбранный элемент словаря — столбец. Эта ориентация должна сохраняться при передаче в алгоритм и при чтении его коэффициентов. Число признаков обычно значительно больше числа слов в одной короткой строке, поскольку словарь объединяет элементы многих документов.

Мешок слов не знает, какой токен является названием темы. Векторизация не использует y и не отличает полезное слово от случайного номера сама по себе. Такое различие задаётся подготовкой, параметрами отбора и проверкой конечной модели.

In [2]:
# Шаг М3.02.02 — Матрица счётчиков на четырёх документах
toy=['доставка заказа задержана','оплата заказа прошла','заказа заказа доставка','нужен возврат оплаты']
count_vectorizer=CountVectorizer()
toy_counts=count_vectorizer.fit_transform(toy)
toy_names=count_vectorizer.get_feature_names_out()
display(pd.DataFrame(toy_counts.toarray(),index=[f'Документ {i}' for i in range(len(toy))],columns=toy_names))
print('Размер:',toy_counts.shape,'ненулевых:',toy_counts.nnz)

,возврат,доставка,задержана,заказа,нужен,оплата,оплаты,прошла
Документ 0,0,1,1,1,0,0,0,0
Документ 1,0,0,0,1,0,1,0,1
Документ 2,0,1,0,2,0,0,0,0
Документ 3,1,0,0,0,1,0,1,0


Размер: (4, 8) ненулевых: 11


**Позиции словаря и ручная проверка**

Индекс в vocabulary_ обозначает позицию, а не частоту или значимость слова. Ручной подсчёт должен использовать тот же анализатор, что и векторизатор.

In [3]:
# Шаг М3.02.03 — Позиции словаря и ручная проверка
analyze=count_vectorizer.build_analyzer()
manual=np.array([[Counter(analyze(text))[term] for term in toy_names] for text in toy])
assert np.array_equal(manual,toy_counts.toarray())
display(pd.Series(count_vectorizer.vocabulary_,name='Позиция').sort_values().to_frame())

,Позиция
возврат,0
доставка,1
задержана,2
заказа,3
нужен,4
оплата,5
оплаты,6
прошла,7


**Частота и бинарное присутствие**

Binary=True сохраняет только наличие элемента. Повтор одного слова перестаёт увеличивать значение. Это другое представление, а не восстановимые сжатые частоты.

In [4]:
# Шаг М3.02.04 — Частота и бинарное присутствие
binary_vectorizer=CountVectorizer(vocabulary=count_vectorizer.vocabulary_,binary=True)
toy_binary=binary_vectorizer.transform(toy)
display(pd.DataFrame(toy_binary.toarray(),columns=toy_names))
print('Счётчик:',toy_counts[2,count_vectorizer.vocabulary_['заказа']])
print('Присутствие:',toy_binary[2,count_vectorizer.vocabulary_['заказа']])

,возврат,доставка,задержана,заказа,нужен,оплата,оплаты,прошла
0,0,1,1,1,0,0,0,0
1,0,0,0,1,0,1,0,1
2,0,1,0,1,0,0,0,0
3,1,0,0,0,1,0,1,0


Счётчик: 2
Присутствие: 1


**Короткие последовательности слов**

N-грамма — последовательность из n соседних элементов. Диапазон (1,2) добавляет пары к одиночным токенам. Он частично сохраняет локальный порядок и увеличивает словарь.

Добавление пар увеличивает число возможных признаков и может сделать каждую конкретную последовательность более редкой. Если пара встретилась только в обучении и никогда не появляется в новых формулировках, она не помогает переносу. Поэтому усложнение словаря не следует автоматически считать улучшением.

При токенизации с ограничением длины односимвольные элементы, например «я», могут исчезать. В данном малом опыте правило изменено явно, чтобы сравнение сохраняло все слова фразы. Это ещё один пример того, почему результаты векторизации нужно проверять на известных минимальных парах.

In [5]:
# Шаг М3.02.05 — Короткие последовательности слов
phrase_pair=['не товар нужен а возврат','товар нужен а не возврат']
unigrams=CountVectorizer(token_pattern=r'(?u)\b\w+\b').fit_transform(phrase_pair)
bigram_vectorizer=CountVectorizer(token_pattern=r'(?u)\b\w+\b',ngram_range=(1,2))
bigrams=bigram_vectorizer.fit_transform(phrase_pair)
print('Одиночные частоты совпали:',np.array_equal(unigrams[0].toarray(),unigrams[1].toarray()))
print('С парами совпали:',np.array_equal(bigrams[0].toarray(),bigrams[1].toarray()))
display(pd.DataFrame(bigrams.toarray(),columns=bigram_vectorizer.get_feature_names_out()))

Одиночные частоты совпали: True
С парами совпали: False


,а,а возврат,а не,возврат,не,не возврат,не товар,нужен,нужен а,товар,товар нужен
0,1,1,0,1,1,0,1,1,1,1,1
1,1,0,1,1,1,1,0,1,1,1,1


**Частота в документах и отсечение**

Document frequency, или df, считает документы с элементом, а не все его появления. Целый min_df задаёт количество документов, дробный — долю. Max_df удаляет слишком распространённые элементы по документной частоте.

In [6]:
# Шаг М3.02.06 — Частота в документах и отсечение
df=np.asarray((toy_counts>0).sum(axis=0)).ravel()
display(pd.DataFrame({'Токен':toy_names,'Всего появлений':np.asarray(toy_counts.sum(axis=0)).ravel(),
                      'Документов с токеном':df}))
filtered=CountVectorizer(min_df=2,max_df=.9).fit(toy)
print('Оставлены:',filtered.get_feature_names_out())

,Токен,Всего появлений,Документов с токеном
0,возврат,1,1
1,доставка,2,2
2,задержана,1,1
3,заказа,4,3
4,нужен,1,1
5,оплата,1,1
6,оплаты,1,1
7,прошла,1,1


Оставлены: ['доставка' 'заказа']


Счётчики и локальные последовательности проверены вручную. Словарь и документные частоты отделены от метки темы.

## 2. Разреженность и стоимость памяти

Разреженная матрица хранит ненулевые значения и позиции. Формат CSR удобен для доступа по документам и поддерживается текстовыми алгоритмами напрямую.

**Устройство CSR на малой матрице**

Data содержит значения, indices — номера столбцов, indptr — границы строк в этих массивах. Nnz обозначает число сохранённых ненулевых элементов.

Строка i хранит позиции от indptr[i] до indptr[i+1], не включая правую границу. Одинаковая пара границ означает строку без ненулевых элементов. В ней не записан полный набор нулей, но форма матрицы всё равно сохраняет ширину словаря.

Число nnz описывает сохранённые значения. Для оценивания памяти также нужны массивы индексов: разреженное хранение не делает служебную адресацию бесплатной. Реальный выигрыш зависит от плотности, типов значений и индексов, а не только от слова sparse в названии объекта.

In [7]:
# Шаг М3.02.07 — Устройство CSR на малой матрице
csr=toy_counts.tocsr()
print('Форма:',csr.shape,'тип:',type(csr).__name__)
print('data:',csr.data)
print('indices:',csr.indices)
print('indptr:',csr.indptr)
left,right=csr.indptr[2:4]
display(pd.DataFrame({'Столбец':csr.indices[left:right],
                      'Токен':toy_names[csr.indices[left:right]],'Значение':csr.data[left:right]}))

Форма: (4, 8) тип: csr_matrix
data: [1 1 1 1 1 1 1 2 1 1 1]
indices: [1 2 3 3 5 7 1 3 0 4 6]
indptr: [ 0  3  6  8 11]


,Столбец,Токен,Значение
0,1,доставка,1
1,3,заказа,2


**Словарь обращений без уплотнения**

Словарь обучается только на train. Число строк сохранено, а ширина определяется токенами и n-граммами. Неизвестные элементы validation не создают столбцы.

In [8]:
# Шаг М3.02.08 — Словарь обращений без уплотнения
corpus_counts=CountVectorizer(preprocessor=normalize_text,ngram_range=(1,2),min_df=2,max_features=12_000)
X_counts_train=corpus_counts.fit_transform(train['text'])
X_counts_valid=corpus_counts.transform(valid['text'])
print('Train:',X_counts_train.shape,'validation:',X_counts_valid.shape)
print('Ненулевых:',X_counts_train.nnz,'доля:',round(X_counts_train.nnz/np.prod(X_counts_train.shape),4))
display(pd.Series(corpus_counts.get_feature_names_out()[:15],name='Имена признаков').to_frame())

Train: (1452, 724) validation: (363, 724)
Ненулевых: 38087 доля: 0.0362


,Имена признаков
0,no
1,no номер
2,адрес
3,адрес для
4,адрес как
5,банк
6,банк отклоняет
7,банк требует
8,банк уже
9,банковской


**Сравнение стоимости хранения**

Оценка CSR складывает размеры трёх массивов. Плотная оценка учитывает каждую позицию. Это стоимость текущего представления, а не пиковая память процесса.

In [9]:
# Шаг М3.02.09 — Сравнение стоимости хранения
csr_bytes=X_counts_train.data.nbytes+X_counts_train.indices.nbytes+X_counts_train.indptr.nbytes
dense_bytes=int(np.prod(X_counts_train.shape))*X_counts_train.dtype.itemsize
display(pd.Series({'CSR, МиБ':csr_bytes/2**20,'Плотная оценка, МиБ':dense_bytes/2**20,
                   'Плотная / CSR':dense_bytes/csr_bytes}).to_frame('Хранение'))
print('100000 документов × 50000 признаков × float64, ГиБ:',100_000*50_000*8/2**30)

,Хранение
"CSR, МиБ",0.441414
"Плотная оценка, МиБ",8.020386
Плотная / CSR,18.169763


100000 документов × 50000 признаков × float64, ГиБ: 37.25290298461914


**Просмотр ненулевых элементов одной строки**

Для диагностики достаточно сохранённых позиций строки. Нормализация и мешок слов уже потеряли часть исходной информации, поэтому исходный текст сохраняется отдельно.

In [10]:
# Шаг М3.02.10 — Просмотр ненулевых элементов одной строки
row=X_counts_train.getrow(0)
names=corpus_counts.get_feature_names_out()
display(pd.DataFrame({'Признак':names[row.indices],'Число':row.data}).sort_values('Число',ascending=False).head(12))
print('Исходный текст:',train['text'].iloc[0])

,Признак,Число
7,заказ,2
0,подскажите,1
2,курьер,1
3,перенёс,1
4,доставку,1
1,пожалуйста,1
5,когда,1
6,привезут,1
8,no,1
9,номер,1


Исходный текст: <p>ПОДСКАЖИТЕ, ПОЖАЛУЙСТА: КУРЬЕР ПЕРЕНЁС ДОСТАВКУ, КОГДА ПРИВЕЗУТ ЗАКАЗ. ЗАКАЗ №83696. НУЖНА ИНФОРМАЦИЯ ПО МОЕМУ ОБРАЩЕНИЮ.</p>


Разреженная форма проверена без выделения большой плотной матрицы. Стоимость хранения отделена от смысла признаков.

## 3. TF-IDF и обучаемые веса

TF-IDF соединяет частоту внутри документа и обратную документную частоту. Формула зависит от параметров. В Scikit-learn по умолчанию используется сглаженный IDF и L2-нормализация строк.

**Ручной расчёт сглаженного IDF**

Для smooth_idf=True используется idf = log((1+n)/(1+df)) + 1. Логарифм натуральный. Добавленные единицы являются частью сглаживания, а не реальными дополнительными документами.

Формула IDF обучается по числу документов с токеном. При сглаживании редкий элемент получает конечный положительный вес. Один и тот же токен может иметь другой IDF на другом обучающем корпусе, поэтому веса нельзя переносить отдельно от соответствующего словаря.

Значения IDF не являются вероятностями и могут превышать единицу. Более редкая опечатка тоже способна получить большой вес. Поэтому просмотр самых больших значений без исходных строк не гарантирует обнаружения содержательных ключевых слов.

In [11]:
# Шаг М3.02.11 — Ручной расчёт сглаженного IDF
manual_idf=np.log((1+len(toy))/(1+df))+1
idf_transformer=TfidfTransformer(norm=None,smooth_idf=True)
toy_weighted=idf_transformer.fit_transform(toy_counts)
display(pd.DataFrame({'Токен':toy_names,'df':df,'IDF вручную':manual_idf,
                      'IDF библиотеки':idf_transformer.idf_}).round(4))
assert np.allclose(manual_idf,idf_transformer.idf_)

,Токен,df,IDF вручную,IDF библиотеки
0,возврат,1,1.9163,1.9163
1,доставка,2,1.5108,1.5108
2,задержана,1,1.9163,1.9163
3,заказа,3,1.2231,1.2231
4,нужен,1,1.9163,1.9163
5,оплата,1,1.9163,1.9163
6,оплаты,1,1.9163,1.9163
7,прошла,1,1.9163,1.9163


**Частота внутри документа и произведение**

В данном варианте TF — исходное число появлений. До нормализации вес равен TF, умноженному на IDF. Предварительного скрытого деления на длину документа нет.

In [12]:
# Шаг М3.02.12 — Частота внутри документа и произведение
manual_weights=toy_counts.toarray()*manual_idf
assert np.allclose(manual_weights,toy_weighted.toarray())
display(pd.DataFrame(manual_weights,columns=toy_names).round(3))

,возврат,доставка,задержана,заказа,нужен,оплата,оплаты,прошла
0,0.000,1.511,1.916,1.223,0.000,0.000,0.000,0.000
1,0.000,0.000,0.000,1.223,0.000,1.916,0.000,1.916
2,0.000,1.511,0.000,2.446,0.000,0.000,0.000,0.000
3,1.916,0.000,0.000,0.000,1.916,0.000,1.916,0.000


**Нормализация вектора документа**

L2-нормализация делит строку на квадратный корень суммы квадратов весов. Это отличается от StandardScaler по столбцам. Пустая строка остаётся нулевой.

Нормализованные документы сопоставимы по длине вектора, но не становятся одинаковыми по значению. Два документа с разными направлениями по-прежнему различаются. Удвоение всех исходных частот без других изменений может исчезнуть после нормализации.

Норма вычисляется только для данной строки, тогда как IDF хранит состояние всего обучающего корпуса. Эти операции имеют разное происхождение. Для transform нового документа используется сохранённый IDF, а его собственная строковая нормировка вычисляется по его ненулевым весам.

In [13]:
# Шаг М3.02.13 — Нормализация вектора документа
normalized_transformer=TfidfTransformer(norm='l2')
toy_tfidf=normalized_transformer.fit_transform(toy_counts)
manual_norm=manual_weights/np.linalg.norm(manual_weights,axis=1,keepdims=True)
assert np.allclose(manual_norm,toy_tfidf.toarray())
display(pd.DataFrame(toy_tfidf.toarray(),columns=toy_names).round(3))
print('Нормы:',np.sqrt(toy_tfidf.multiply(toy_tfidf).sum(axis=1)).A1)

,возврат,доставка,задержана,заказа,нужен,оплата,оплаты,прошла
0,0.000,0.553,0.702,0.448,0.000,0.000,0.000,0.000
1,0.000,0.000,0.000,0.411,0.000,0.645,0.000,0.645
2,0.000,0.525,0.000,0.851,0.000,0.000,0.000,0.000
3,0.577,0.000,0.000,0.000,0.577,0.000,0.577,0.000


Нормы: [1. 1. 1. 1.]


**Ослабление повторов через логарифм**

Sublinear_tf заменяет положительную частоту на 1 + log(tf). Повтор усиливает вес медленнее. Нормализация отключена, чтобы наблюдать одно изменение.

In [14]:
# Шаг М3.02.14 — Ослабление повторов через логарифм
sublinear=TfidfTransformer(norm=None,sublinear_tf=True)
sublinear_weights=sublinear.fit_transform(toy_counts)
position=count_vectorizer.vocabulary_['заказа']
display(pd.DataFrame({'Документ':[1,2],
                      'TF':[toy_counts[i,position] for i in [1,2]],
                      'Обычный вес':[toy_weighted[i,position] for i in [1,2]],
                      'Sublinear':[sublinear_weights[i,position] for i in [1,2]]}).round(4))

,Документ,TF,Обычный вес,Sublinear
0,1,1,1.2231,1.2231
1,2,2,2.4463,2.0710


**Общий TF-IDF только по train**

TfidfVectorizer объединяет счётчики и веса. Validation проходит только transform, без изменения словаря и IDF. Выход остаётся разреженным.

In [15]:
# Шаг М3.02.15 — Общий TF-IDF только по train
tfidf=TfidfVectorizer(preprocessor=normalize_text,ngram_range=(1,2),min_df=2,
                       max_features=12_000,sublinear_tf=True)
X_tfidf_train=tfidf.fit_transform(train['text'])
X_tfidf_valid=tfidf.transform(valid['text'])
print('Формы:',X_tfidf_train.shape,X_tfidf_valid.shape)
print('Формат:',type(X_tfidf_train).__name__)
print('Диапазон IDF:',round(tfidf.idf_.min(),3),round(tfidf.idf_.max(),3))

Формы: (1452, 724) (363, 724)
Формат: csr_matrix
Диапазон IDF: 1.0 7.183


**Неизвестный текст и нулевой вектор**

OOV обозначает элементы вне обученного словаря. Нулевой выход возможен у пустого текста, текста без подходящих токенов и полностью неизвестной лексики. Эти причины различаются до классификатора.

Нулевое представление не позволяет отличить несколько полностью неизвестных сообщений. Если оба дали только нули, классификатор получает одинаковый вход. Разница исходных смыслов не может появиться у него из отсутствующих координат.

В рабочем применении полезно сохранять две отдельные диагностики: пустоту исходного текста и наличие известных элементов после подготовки. Эти сигналы позволяют направить запрос на уточнение вместо выдачи технически возможного, но необоснованного автоматического ответа.

In [16]:
# Шаг М3.02.16 — Неизвестный текст и нулевой вектор
unknown_examples=['','я','фжлхщц ззюэщ','доставка','Оплата заказа']
unknown_matrix=tfidf.transform(unknown_examples)
display(pd.DataFrame({'Текст':unknown_examples,
                      'Ненулевых признаков':unknown_matrix.getnnz(axis=1),
                      'Пустая строка':[not s.strip() for s in unknown_examples]}))

,Текст,Ненулевых признаков,Пустая строка
0,,0,True
1,я,0,False
2,фжлхщц ззюэщ,0,False
3,доставка,1,False
4,Оплата заказа,1,False


**Утечка частот без передачи меток**

Игрушечный опыт показывает изменение представления при fit на объединении частей. Неправильный объект не используется в основном эксперименте.

In [17]:
# Шаг М3.02.17 — Утечка частот без передачи меток
past=['нужна доставка заказа','оплата заказа подтверждена']
future=['редчайшееслово появилось позже']
honest=TfidfVectorizer().fit(past)
leaky=TfidfVectorizer().fit(past+future)
print('Токен в честном словаре:','редчайшееслово' in honest.vocabulary_)
print('Токен после подглядывания:','редчайшееслово' in leaky.vocabulary_)
print('Вес общего слова:',honest.idf_[honest.vocabulary_['заказа']],
      leaky.idf_[leaky.vocabulary_['заказа']])

Токен в честном словаре: False
Токен после подглядывания: True
Вес общего слова: 1.0 1.2876820724517808


TF-IDF проверен вручную и применён без обучения на validation. Нулевые документы и варианты формулы обозначены явно.

## 4. Символьные признаки и фиксированное пространство

Символьные фрагменты сохраняют часть общих элементов при опечатке. Feature hashing фиксирует число координат без словаря, но допускает столкновения разных элементов.

**Символьные n-граммы и опечатка**

Char_wb выделяет фрагменты внутри границ слова с добавлением граничных пробелов. Это не исправление текста и не морфологический разбор. Похожие буквы не гарантируют одинаковый смысл.

Символьные фрагменты чувствительны к своему диапазону длин. Короткие элементы чаще совпадают у разных слов, длинные более специфичны и легче разрушаются опечаткой. Размер словаря также зависит от дефисов, пробелов и выбранной нормализации.

Режим char_wb использует границы слов, поэтому отличается от общего char, который может выделять последовательности через пробелы. Сравнение этих вариантов относится к представлению. Только последующий эксперимент покажет, помогает ли оно на нужных формулировках и при каких затратах.

In [18]:
# Шаг М3.02.18 — Символьные n-граммы и опечатка
word_toy=TfidfVectorizer().fit(['доставка','оплата'])
char_toy=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5)).fit(['доставка','оплата'])
comparison_text=['доставка','даставка']
word_rows=word_toy.transform(comparison_text)
char_rows=char_toy.transform(comparison_text)
display(pd.DataFrame({'Текст':comparison_text,'Словных признаков':word_rows.getnnz(axis=1),
                      'Символьных признаков':char_rows.getnnz(axis=1)}))
print('Примеры фрагментов:',char_toy.get_feature_names_out()[:12])

,Текст,Словных признаков,Символьных признаков
0,доставка,1,21
1,даставка,0,12


Примеры фрагментов: [' до' ' дос' ' дост' ' оп' ' опл' ' опла' 'авк' 'авка' 'авка ' 'ата'
 'ата ' 'вка']


**Стоимость символьного представления**

Вход и разбиение те же. Ограничение max_features задано до сравнения классификаторов. Здесь измеряется представление, а не точность темы.

In [19]:
# Шаг М3.02.19 — Стоимость символьного представления
char_vectorizer=TfidfVectorizer(preprocessor=normalize_text,analyzer='char_wb',ngram_range=(3,5),
                                min_df=2,max_features=12_000,sublinear_tf=True)
X_char=char_vectorizer.fit_transform(train['text'])
display(pd.DataFrame({'Представление':['Словное TF-IDF','Символьное TF-IDF'],
                      'Признаков':[X_tfidf_train.shape[1],X_char.shape[1]],
                      'Ненулевых':[X_tfidf_train.nnz,X_char.nnz],
                      'CSR, МиБ':[(m.data.nbytes+m.indices.nbytes+m.indptr.nbytes)/2**20
                                  for m in [X_tfidf_train,X_char]]}))

,Представление,Признаков,Ненулевых,"CSR, МиБ"
0,Словное TF-IDF,724,38087,0.441414
1,Символьное TF-IDF,2934,293841,3.368286


**Порции с одним обученным словарём**

Отдельный transform сохраняет координаты. Разреженный vstack объединяет строки. Новый fit на каждой порции создал бы несогласованные словари.

In [20]:
# Шаг М3.02.20 — Порции с одним обученным словарём
pieces=[tfidf.transform(valid['text'].iloc[start:start+100]) for start in range(0,len(valid),100)]
stacked=sparse.vstack(pieces,format='csr')
delta=stacked-X_tfidf_valid
assert delta.nnz==0 or np.allclose(delta.data,0)
print('Порций:',len(pieces),'форма:',stacked.shape)

Порций: 4 форма: (363, 724)


**Feature hashing без обученного словаря**

HashingVectorizer направляет элементы в заранее заданное число координат. У него нет выученных vocabulary_ и IDF. Число признаков задано заранее.

При фиксированном пространстве добавление нового документа не меняет смысл позиции по вычислительному правилу. При этом одна позиция может объединять несколько токенов. Такая неоднозначность заложена заранее и не исправляется последующим обратным просмотром имени столбца.

Отключение alternate_sign обеспечивает неотрицательность, но меняет статистическое поведение столкновений по сравнению со знаковым вариантом. Это инженерный выбор под требования дальнейшего алгоритма. Подбор размера пространства выполняется с учётом памяти и качества, а не ради полного устранения любых коллизий.

In [21]:
# Шаг М3.02.21 — Feature hashing без обученного словаря
hash_vectorizer=HashingVectorizer(preprocessor=normalize_text,n_features=2**12,
                                alternate_sign=False,norm=None)
hash_batch=hash_vectorizer.transform(train['text'].head(100))
print('Форма:',hash_batch.shape,'ненулевых:',hash_batch.nnz)
print('Есть словарь:',hasattr(hash_vectorizer,'vocabulary_'))
print('Есть IDF:',hasattr(hash_vectorizer,'idf_'))

Форма: (100, 4096) ненулевых: 1382
Есть словарь: False
Есть IDF: False


**Столкновения в маленьком пространстве**

Коллизия означает совпадение координаты разных токенов. Число занятых корзин не равно размеру исходного словаря. Малое пространство выбрано для наглядного опыта.

In [22]:
# Шаг М3.02.22 — Столкновения в маленьком пространстве
single_terms=[term for term in corpus_counts.get_feature_names_out() if ' ' not in term][:120]
collision_rows=[]
for width in [8,128,4096]:
    small_hash=HashingVectorizer(n_features=width,alternate_sign=False,norm=None)
    term_matrix=small_hash.transform(single_terms)
    occupied=len(np.unique(term_matrix.indices))
    collision_rows.append({'Координат':width,'Токенов':len(single_terms),
                            'Занято':occupied,'Потеря различимости':len(single_terms)-occupied})
display(pd.DataFrame(collision_rows))

,Координат,Токенов,Занято,Потеря различимости
0,8,120,8,112
1,128,120,80,40
2,4096,120,119,1


**Последовательная обработка исходного TXT**

Накопитель содержит только 4096 чисел. Порции преобразуются и освобождаются; вся матрица корпуса не сохраняется. Итоги относятся к корзинам, а не восстановленным словам.

In [23]:
# Шаг М3.02.23 — Последовательная обработка исходного TXT
totals=np.zeros(2**12,dtype=float)
buffer=[]
processed=0
with (course_data/'reviews.txt').open(encoding='utf-8') as stream:
    for line in stream:
        buffer.append(line.rstrip(chr(10)))
        if len(buffer)==2000:
            block=hash_vectorizer.transform(buffer)
            totals+=np.asarray(block.sum(axis=0)).ravel()
            processed+=len(buffer)
            buffer=[]
    if buffer:
        totals+=np.asarray(hash_vectorizer.transform(buffer).sum(axis=0)).ravel()
        processed+=len(buffer)
print('Документов:',processed,'координат:',totals.size)
print('Занятых корзин:',np.count_nonzero(totals),'сумма счётчиков:',int(totals.sum()))
assert processed==50_000

Документов: 50000 координат: 4096
Занятых корзин: 20 сумма счётчиков: 449221


Рассмотрены устойчивость к форме слова и ограниченная по размеру схема. Разреженность, словарь и hashing выполняют разные роли.

## 5. Чтение признаков и итоговые проверки

Большой вес TF-IDF является относительной характеристикой документа по выбранной формуле. Он не равен вероятности класса и не доказывает смысл. Сохраняется связь строки с исходным обращением.

**Большие веса отдельного обращения**

Показаны веса одного документа до обучения классификатора. Они не являются важностью для темы. Сумма весов не обязана равняться единице.

In [24]:
# Шаг М3.02.24 — Большие веса отдельного обращения
row=X_tfidf_train.getrow(0)
names=tfidf.get_feature_names_out()
weights=pd.Series(row.data,index=names[row.indices]).sort_values(ascending=False)
display(weights.head(12).to_frame('TF-IDF'))
print('Документ:',train['text'].iloc[0])
print('Сумма:',round(float(row.sum()),4),'L2-норма:',round(float(np.sqrt(row.multiply(row).sum())),4))

,TF-IDF
пожалуйста курьер,0.326029
перенёс,0.242812
привезут,0.242812
доставку когда,0.242812
курьер перенёс,0.242812
курьер,0.242812
перенёс доставку,0.242812
когда привезут,0.242812
привезут заказ,0.242812
когда,0.223125


Документ: <p>ПОДСКАЖИТЕ, ПОЖАЛУЙСТА: КУРЬЕР ПЕРЕНЁС ДОСТАВКУ, КОГДА ПРИВЕЗУТ ЗАКАЗ. ЗАКАЗ №83696. НУЖНА ИНФОРМАЦИЯ ПО МОЕМУ ОБРАЩЕНИЮ.</p>
Сумма: 5.0687 L2-норма: 1.0


**Обратное получение элементов**

Inverse_transform возвращает имена ненулевых элементов. Частоты, полный порядок, регистр и исходные номера таким действием не восстанавливаются.

Список ненулевых признаков содержит только элементы обученного словаря. Неизвестные слова туда не попадут, даже если они определяли смысл исходного обращения. Поэтому короткий список не является сокращённым пересказом документа.

Для последующего разбора сохраняется согласованность трёх объектов: исходной строки, её числового представления и вывода алгоритма. Идентификатор связывает их, но не обязательно входит в X. Это позволяет объяснять ошибки, не объявляя технические номера содержательными признаками.

In [25]:
# Шаг М3.02.25 — Обратное получение элементов
back=tfidf.inverse_transform(X_tfidf_train[:2])
display(pd.DataFrame({'Исходное':train['text'].iloc[:2].tolist(),
                      'Ненулевые элементы':[list(x[:12]) for x in back]}))

,Исходное,Ненулевые элементы
0,"<p>ПОДСКАЖИТЕ, ПОЖАЛУЙСТА: КУРЬЕР ПЕРЕНЁС ДОСТ...","[подскажите, пожалуйста, курьер, перенёс, дост..."
1,"Здравствуйте. Курьер перенёс доставку, когда п...","[курьер, перенёс, доставку, когда, привезут, з..."


**Паспорт числового представления**

Контроль формы и значений выполняется до оценки конечной задачи. Классификатору и поисковой процедуре можно передать CSR напрямую.

In [26]:
# Шаг М3.02.26 — Паспорт числового представления
checks=pd.Series({
    'Строки train сохранены':X_tfidf_train.shape[0]==len(train),
    'Строки validation сохранены':X_tfidf_valid.shape[0]==len(valid),
    'Схема одинакова':X_tfidf_train.shape[1]==X_tfidf_valid.shape[1],
    'Матрица разреженная':sparse.issparse(X_tfidf_train),
    'Веса конечны':np.isfinite(X_tfidf_train.data).all(),
    'Веса неотрицательны':(X_tfidf_train.data>=0).all(),
    'Имена соответствуют':len(tfidf.get_feature_names_out())==X_tfidf_train.shape[1]
},name='Проверено')
display(checks.to_frame())
assert checks.all()

,Проверено
Строки train сохранены,True
Строки validation сохранены,True
Схема одинакова,True
Матрица разреженная,True
Веса конечны,True
Веса неотрицательны,True
Имена соответствуют,True


Представления прочитаны на уровне документа и проверены без потери связи с источником.

## Полученный результат

Проверены мешок слов, n-граммы, сглаженный TF-IDF и нормализация. Изучены CSR, неизвестные токены, символьные признаки и ограниченное потоковое представление. Обучаемые состояния рассчитаны по train.

Далее сравниваются классификаторы, а затем рассматриваются сходство, поиск и группировка документов.

## Документация

[Извлечение признаков](https://scikit-learn.org/1.8/modules/feature_extraction.html#text-feature-extraction)

[TfidfVectorizer](https://scikit-learn.org/1.8/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

[HashingVectorizer](https://scikit-learn.org/1.8/modules/generated/sklearn.feature_extraction.text.HashingVectorizer.html)